In [ ]:
import sys
from pathlib import Path
import pandas as pd
import os
import re
sys.path.insert(0, '.')
from utils import DATA_DIR

departure_dir = DATA_DIR / 'ocha_monthly_departures'
returnee_dir  = DATA_DIR / 'ocha_monthly_returnees'

eastern_provinces = ['Nord-kivu', 'Sud-kivu', 'Ituri']

month_map = {
    'janvier': '01', 'fevrier': '02', 'fev': '02', 'ferv': '02',
    'mars': '03', 'avril': '04', 'mai': '05', 'juin': '06',
    'juillet': '07', 'aout': '08', 'septembre': '09',
    'octobre': '10', 'novembre': '11', 'decembre': '12'
}

usecols = ['id', 'movement_date', 'evaluation_date', 'household', 'person',
           'population_id', 'population_label', 'cause_label',
           'admin1_id', 'admin1_label', 'admin2_label',
           'admin1_id_from', 'admin1_label_from']

def extract_snapshot_month(filename):
    filename = filename.lower()
    year_match = re.search(r'(20\d{2})', filename)
    if not year_match:
        return None
    year = year_match.group(1)
    for month_fr, month_num in month_map.items():
        if month_fr in filename:
            return f"{year}-{month_num}"
    return None

def get_sheet_name(xl):
    valid_sheets = [
        'Data', 'Data2', 'DATA', 'Data (2)',
        'IDPs', 'Idps',
        'Retournées', 'Retournée', 'Retournee', 'Retournés', 'Retour',
        'Déplacées',
        'Feuil1', 'sheet',
        'Sheet1', 'Sheet2',
        'factsheet_mai_2023', 'factsheet_mai',
        'Facteesheet-janv2023', 'Factsheet janv 2023',
        'Factsheet-Octobre2021',
        '01_source'
    ]
    for name in valid_sheets:
        if name in xl.sheet_names:
            return name
    return None

def open_excel(filepath):
    """Try openpyxl first, fall back to xlrd for old-format files."""
    for engine in ['openpyxl', 'xlrd']:
        try:
            return pd.ExcelFile(filepath, engine=engine), engine
        except Exception:
            continue
    return None, None

def load_files(directory):
    all_dfs = []
    all_files = os.listdir(directory)

    files_to_skip = [
        f for f in all_files
        if not f.endswith('__1_.xlsx')
        and f.replace('.xlsx', '__1_.xlsx') in all_files
    ]

    for filename in sorted(all_files):
        if not filename.endswith('.xlsx'):
            continue
        if filename in files_to_skip:
            print(f"  Skipping (superseded): {filename}")
            continue

        snapshot_month = extract_snapshot_month(filename)
        if not snapshot_month:
            print(f"  Skipping (no date found): {filename}")
            continue

        filepath = os.path.join(directory, filename)

        xl, engine = open_excel(filepath)
        if xl is None:
            print(f"  Skipping (unreadable format): {filename}")
            continue

        sheet_name = get_sheet_name(xl)
        if not sheet_name:
            print(f"  Skipping (no valid sheet): {filename}")
            continue

        try:
            df = pd.read_excel(
                filepath,
                sheet_name=sheet_name,
                engine=engine,
                usecols=lambda x: x in usecols
            )
        except Exception as e:
            print(f"  Skipping (read error): {filename} | {e}")
            continue

        if 'admin1_label' not in df.columns or 'person' not in df.columns:
            print(f"  Skipping (missing required columns): {filename}")
            continue

        df['snapshot_month'] = snapshot_month
        df['source_file'] = filename
        all_dfs.append(df)
        print(f"  Loaded: {filename} -> {snapshot_month} ({len(df)} rows)")

    return pd.concat(all_dfs, ignore_index=True)

# Load departures
print("Loading departure files...")
df_departures = load_files(departure_dir)
print(f"\nTotal departure rows: {len(df_departures)}")

# Load returnees
print("\nLoading returnee files...")
df_returnees = load_files(returnee_dir)
print(f"\nTotal returnee rows: {len(df_returnees)}")


In [ ]:
# Save departees and returnees as separate CSVs
df_departures.to_csv(DATA_DIR / 'departees_eastern_drc.csv', index=False)
df_returnees.to_csv(DATA_DIR / 'returnees_eastern_drc.csv', index=False)

print(f"Departees saved: {len(df_departures):,} rows")
print(f"Returnees saved: {len(df_returnees):,} rows")


In [ ]:
# ── Helper: aggregate event-level rows to province-month panel ────────────
def build_province_panel(df_dep, df_ret, eastern_provinces):
    """Filter, deduplicate, and aggregate departure/returnee DataFrames
    to a province × month panel with net_monthly_flow.

    Parameters
    ----------
    df_dep, df_ret : raw event-level DataFrames with snapshot_month and movement_date
    eastern_provinces : list of province names to keep

    Returns
    -------
    DataFrame with columns: admin1_label, snapshot_month,
        total_displaced, num_sites_displaced, total_returnees,
        num_sites_returnees, net_monthly_flow
    """
    def filter_and_dedup(df):
        df = df.copy()
        df['movement_date']   = pd.to_datetime(df['movement_date'],   errors='coerce')
        df['snapshot_month_dt'] = pd.to_datetime(df['snapshot_month'])

        # Keep only rows where movement_date falls within the snapshot month
        df = df[
            (df['admin1_label'].isin(eastern_provinces)) &
            (df['movement_date'].dt.year  == df['snapshot_month_dt'].dt.year) &
            (df['movement_date'].dt.month == df['snapshot_month_dt'].dt.month)
        ]
        # One row per event — keep earliest snapshot_month
        return df.sort_values('snapshot_month').drop_duplicates(subset=['id'], keep='first')

    dep_clean = filter_and_dedup(df_dep)
    ret_clean = filter_and_dedup(df_ret)
    print(f'Departures after filter/dedup: {len(dep_clean):,}')
    print(f'Returnees  after filter/dedup: {len(ret_clean):,}')

    dep_panel = (
        dep_clean
        .groupby(['admin1_label', 'snapshot_month'], as_index=False)
        .agg(total_displaced=('person', 'sum'), num_sites_displaced=('id', 'nunique'))
        .sort_values(['admin1_label', 'snapshot_month'])
    )
    ret_panel = (
        ret_clean
        .groupby(['admin1_label', 'snapshot_month'], as_index=False)
        .agg(total_returnees=('person', 'sum'), num_sites_returnees=('id', 'nunique'))
        .sort_values(['admin1_label', 'snapshot_month'])
    )

    print(f'\nPre-merge  — dep panel rows: {len(dep_panel):,}')
    panel = pd.merge(dep_panel, ret_panel,
                     on=['admin1_label', 'snapshot_month'], how='outer').fillna(0)
    panel['net_monthly_flow'] = panel['total_displaced'] - panel['total_returnees']
    panel = panel.sort_values(['admin1_label', 'snapshot_month'])
    print(f'Post-merge — final panel rows: {len(panel):,}')
    return panel


# eastern_provinces already defined in the first cell
df_panel = build_province_panel(df_departures, df_returnees, eastern_provinces)

print(f'\nTimepoints per province:')
print(df_panel.groupby('admin1_label')['snapshot_month'].count().sort_values(ascending=False).to_string())
print(f'\nDate range per province:')
print(df_panel.groupby('admin1_label')['snapshot_month'].agg(['min', 'max']).to_string())


In [ ]:
# ── Save ───────────────────────────────────────────────────────────────────
OUT_PATH = DATA_DIR / 'idp_dat_eastern_drc.csv'
df_panel.to_csv(OUT_PATH, index=False)
print(f'Saved {len(df_panel):,} rows → {OUT_PATH}')
